## 01 · W1 loss weights only

Train a fresh four-loss model on the baseline COCO positives, changing only MIR and MLM coefficients.

Notebook outline only: markdown cells describe the planned Colab implementation. No training code, data manifests, measurements, checkpoints, or scores are included. The experiment has not been run.

Reference: [MACCO source at `97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae`](https://github.com/hiker-lw/MACCO/tree/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae). The author code is under the [MACCO MIT license](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/LICENSE). When implementing, preserve original notices and keep the needed code visible in this notebook.


## S01 · Environment and assets

Record the GPU, CUDA availability, PyTorch and torchvision versions, and required packages (`ftfy`, `regex`, and the dependencies used by the selected author code). Stage chosen images under `/content/macco_data/`. Obtain the CLIP tokenizer vocabulary and original OpenAI ViT-B/32 weights as data assets. Keep implementation inside this notebook when code is added.


## S02 · Tokenizer

Use the pinned MACCO `src/open_clip_code/tokenizer.py` as the reference for the CLIP BPE helpers, `SimpleTokenizer`, and `tokenize`. Record an explicit path to `bpe_simple_vocab_16e6.txt.gz` and check positive and reviewed-negative captions use the same tokenizer.


## S03 · CLIP and MACCO model

Use ViT-B/32 with QuickGELU, the pinned CLIP and MACCO model definitions, and both image and text encoders. For every independent run, reload the same original OpenAI CLIP checkpoint, then initialize new MACCO predictors with the recorded common seed. Record the checkpoint identifier and hash. Train both encoders from the first optimizer step; use float32 trainable parameters with AMP.


## S04 · Preprocessing and datasets

Preserve the author's image transforms, crop-aware box-to-patch conversion, text masking, annotation fields, and joint image–text path. Select one positive row per source image or frame. Freeze row IDs before training. Shared COCO counts: 1,000 train from `train2014`, 500 project validation and 500 project test from disjoint `val2014` rows. Shared Talk2Car counts: 1,000 train, 500 official labeled validation, and 500 project test from whole scenes reserved from the official labeled training set. Check image, frame, and scene overlap.


## S05 · Original losses

Use the pinned author definitions, masking, normalization, targets, and reductions for MCA, MIR, MLM, and MIM. Baseline coefficients: MCA 1.0, MIR 0.1, MLM 20.0, MIM 250.0. Compare the four unweighted loss values on one fixed batch with the pinned author implementation during development.


## S06 · Training and checkpoints

Use the pinned single-GPU joint training path with a fresh optimizer per run, AMP, one update per batch, and `drop_last=False`. Train five full epochs over 1,000 positive pairs (5,000 pair presentations). Learning rates: `5e-7` for pretrained encoders and `1e-3` for new predictors; warmup: 50 updates. Choose one common batch size after the separate 8/16/32 speed check; candidate 16 gives 63 updates per epoch and 315 total. Compute the final update budget from the loader length. Save resolved settings, per-term losses, rows per epoch, update count, and checkpoint path.


## S07 · Shared evaluation

Freeze cases, candidate captions, preprocessing, scoring rules, and noise seed before final testing. Use the same 500-row Talk2Car validation and test pools for all five runs; COCO-trained runs also use the same COCO validation and test pools and separate selected Table 1 and Table 21 evaluations. Report image-to-text and text-to-image Recall@1, Recall@5, and Recall@10 over fixed candidate pools. Score clean and fixed-noise inputs on the same cases. Record benchmark, split, condition, seed, metric, unit, candidate pool, and `n_examples`. Compare variants with baseline only after matching these keys.


## 01.01 · Setup and shared inputs

Use this notebook's S01–S07 implementation and load the frozen COCO splits, baseline settings, and `baseline_metrics.csv`. Match model architecture, original OpenAI initialization, predictor seed, batch size, and five-epoch budget.


## 01.02 · W1 weights

Set MCA (`contrast_loss_w`) to 1.0, MIR (`intra_modal_contrast_loss_w`) to 0.2, MLM (`mask_text_loss_w`) to 25.0, and MIM (`mask_image_loss_w`) to 250.0. Compare resolved settings with baseline; MIR and MLM are the two changes.


## 01.03 · One-batch loss check

Compute the four unweighted terms and confirm total loss equals `MCA + 0.2 × MIR + 25.0 × MLM + 250.0 × MIM`.


## 01.04 · Independent training

Reload original OpenAI CLIP weights; initialize fresh predictors and optimizer. Train five full epochs over the same 1,000 COCO pairs. Save resolved settings, four-term loss log, rows, update count, and checkpoint path.


## 01.05 · Evaluate and compare

Evaluate on the shared frozen cases with the shared functions. Match benchmark, split, metric, condition, seed, and case count before computing `W1 − baseline`. Save `w1_comparison.csv`; accuracy percentage deltas use percentage points.
